In [1]:
import pandas as pd

train = pd.read_csv("../data/train_features.csv")
X = train.drop(columns = ["price", "id"]).astype(float)
y = train["price"]
n = len(train)
bins = []

for i in range(5):
    bins.append([round(i * n / 5), round((i + 1) * n / 5)])

for i in range(5):
    print(f"Grupo {i}, de {bins[i][0]} a {bins[i][1]}, con longitud {bins[i][1] - bins[i][0]}")

assert sum([bins[i][1] - bins[i][0] for i in range(5)]) == n

Grupo 0, de 0 a 2549, con longitud 2549
Grupo 1, de 2549 a 5099, con longitud 2550
Grupo 2, de 5099 a 7648, con longitud 2549
Grupo 3, de 7648 a 10198, con longitud 2550
Grupo 4, de 10198 a 12747, con longitud 2549


In [2]:
import numpy as np

rmses = []
maes = []
r2s = []

for j in range(5):

    X_va = X.iloc[bins[j][0] : bins[j][1]]
    X_tr = pd.concat([X.iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)])

    X_tr_mean = X_tr.mean()
    X_tr_std = X_tr.std()

    assert len(X_tr_mean) == 132

    X_va = ((X_va - X_tr_mean) / X_tr_std).to_numpy()
    X_tr = ((X_tr - X_tr_mean) / X_tr_std).to_numpy()
    y_va = y.iloc[bins[j][0] : bins[j][1]].to_numpy()
    y_tr = pd.concat([y.iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)]).to_numpy()
    w = np.zeros(X_tr.shape[1])
    b = 0
    lr = 0.1
    n = len(X_tr)

    assert len(X_va) + len(X_tr) == len(X)

    assert train["id"].iloc[bins[j][0] : bins[j][1]].isin(pd.concat([train["id"].iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)]).to_numpy()).sum() == 0

    for _ in range(1000):
        y_hat = X_tr @ w + b
        error = y_tr - y_hat
        dw = - 2 / n * X_tr.T @ error
        db = - 2 * np.mean(error)
        w -= lr * dw
        b -= lr * db

    y_hat = X_va @ w + b
    error = y_va - y_hat
    mse = np.mean(error ** 2)
    rmse = mse ** 0.5
    rmses.append(rmse)
    mae = np.mean(abs(error))
    maes.append(mae)
    r2 = 1 - mse / (y_va.std() ** 2)
    r2s.append(r2)

print(rmses)
print(np.mean(rmses))
print(np.std(rmses))
print(maes)
print(np.mean(maes))
print(np.std(maes))
print(r2s)
print(np.mean(r2s))
print(np.std(r2s))


[np.float64(115.39567745009653), np.float64(115.37241415013311), np.float64(123.88088827711444), np.float64(107.77979839411371), np.float64(133.53880503452322)]
119.19351666119618
8.798364881424032
[np.float64(62.45177026613717), np.float64(61.63503359955312), np.float64(62.42210600863206), np.float64(58.65049292228991), np.float64(62.80741454605266)]
61.593363468532985
1.5205143799736915
[np.float64(0.32293109926108876), np.float64(0.37280385983816344), np.float64(0.3541486695880467), np.float64(0.3752899486357789), np.float64(0.326061192605831)]
0.35024695398578176
0.022281364142181265


In [3]:
fold4 = pd.DataFrame({"price": y_va, "pred": y_hat, "error": error, "abs_error": abs(error)})
fold4 = fold4.sort_values(by="abs_error", ascending=False)
print(fold4.head(10))
fold4_filtered = fold4.iloc[10:]
mse = np.mean(fold4_filtered["error"] ** 2)
rmse = mse ** 0.5
mae = np.mean(abs(fold4_filtered["error"]))

print(rmse)
print(mae)

       price         pred        error    abs_error
351   1712.0   117.040941  1594.959059  1594.959059
283     47.2  1421.226372 -1374.026372  1374.026372
1249  1892.0   559.331898  1332.668102  1332.668102
1699  1892.0   559.796281  1332.203719  1332.203719
2298  1887.0   559.343974  1327.656026  1327.656026
1000  1740.5   458.632120  1281.867880  1281.867880
1167  1501.0   251.062106  1249.937894  1249.937894
2036  1767.5   557.584386  1209.915614  1209.915614
2519   122.0  1228.706529 -1106.706529  1106.706529
1214  1264.5   202.307267  1062.192733  1062.192733
106.29565244788182
57.985019988416916


## Comparing three metrics

Linear regression (gradient descent, 1000 steps, lr = 0.1) evaluated with 5-fold cross-validation on the training set. Each fold is used once for evaluation, with a new model trained on the other four and standardisation statistics computed on those four only.

| Fold | RMSE (€) | MAE (€) | R² |
| ---: | ---: | ---: | ---: |
| 0 | 115.40 | 62.45 | 0.323 |
| 1 | 115.37 | 61.64 | 0.373 |
| 2 | 123.88 | 62.42 | 0.354 |
| 3 | 107.78 | 58.65 | 0.375 |
| 4 | 133.54 | 62.81 | 0.326 |
| **Mean ± std** | **119.19 ± 8.80** | **61.59 ± 1.52** | **0.350 ± 0.022** |

### What each metric measures

- **RMSE** is the square root of the mean squared error. Errors are squared before averaging, so a few large errors weigh much more than many small ones. With three listings with errors of 10, 10 and 1,000 €, RMSE = √((100 + 100 + 1,000,000) / 3) ≈ 577 €.
- **MAE** is the mean absolute error. Each error counts in proportion to its size, so extreme errors still move it, but much less: in the same example, MAE = 1,020 / 3 = 340 €. It is closer to the typical error.
- **R²** is not in euros. It compares the model with a baseline that always predicts the mean, whose MSE is the variance of y: R² = 1 − MSE / std(y)². It is relative to how spread out the prices of each fold are. Fold 2 has a higher RMSE than fold 0 (123.88 vs 115.40 €) but also a higher R² (0.354 vs 0.323), because its prices are more spread out (std ≈ 154 € vs ≈ 140 €).

### How they compare

- RMSE varies much more across folds than MAE (std 8.80 € vs 1.52 €). A few extreme listings, randomly distributed across folds, are enough to push the RMSE of one fold up, while the MAE barely notices them.
- The metrics do not rank the folds the same way. Fold 4 is the worst by RMSE and by MAE (although its MAE is only 0.36 € above fold 0), but fold 0 is the worst by R², because its prices have a smaller variance (std ≈ 140 € vs ≈ 163 € in fold 4).

### A case where a metric misleads: fold 4

By RMSE, fold 4 looks clearly like the worst fold (133.54 € vs a mean of 119.19 €). But that comes from only 10 of its 2,549 listings (0.4 %), each with an error above 1,000 €. Removing them and recomputing:

| Fold 4 | RMSE (€) | MAE (€) |
| --- | ---: | ---: |
| All 2,549 listings | 133.54 | 62.81 |
| Without the 10 largest errors | 106.30 | 57.99 |

Without them, fold 4 has the best RMSE of all folds, while its MAE only drops by about 5 €. The model is not worse on fold 4 in general; a handful of listings dominate the metric.

Those 10 errors are of two kinds:

- **Very expensive listings predicted far too low** (around 1,700–1,900 € predicted at 100–560 €). There are very few listings at those prices, and the model does not predict values that high.
- **Cheap listings predicted far too high**, probably because of errors in the data: a listing for 1 guest with 50 bedrooms (47.20 €, predicted 1,421 €) and one for 2 guests with 38 bathrooms (122 €, predicted 1,229 €). A linear model multiplies these values by their weights, so the prediction explodes. In this fold they were not seen during training, but even if they had been, a linear model cannot learn that such values are errors.

### Conclusion

If I had to report a single metric, I would use MAE, because it is the closest to the typical error. Next to it I would report RMSE: being in the same units, an RMSE much larger than the MAE (119 vs 62 € here) shows that there are a few very large errors and that the model is not always reliable.